In [0]:
df=spark.read.csv("/Volumes/dev/naval/raw/customers.csv",header=True,inferSchema=True)

In [0]:
df.show()

In [0]:
df.display()

In [0]:
df.select("customer_id", "name")

In [0]:
df.write.mode("overwrite").saveAsTable("dev.aryan.customers_pyspark")

In [0]:
df = spark.read.parquet("/Volumes/dev/naval/raw/part-00000-4ece8039-e05f-4114-8262-9d82c9ee1e62-c000.snappy.parquet")

In [0]:
df.write.mode("overwrite").saveAsTable("dev.aryan.trans_pyspark")

In [0]:
input_path="/Volumes/dev/naval/raw/products.csv"
catalog="dev"
schema="aryan"
tablname="product_pyspark"

In [0]:
df=spark.read.csv(input_path,header=True,inferSchema=True)
df.write.mode('overwrite').saveAsTable(f"{catalog}.{schema}.{tablname}")

In [0]:
from pyspark.sql.functions import current_timestamp, col

df1=df.withColumn("ingestion_date", current_timestamp()).withColumn("path", col("_metadata.file_path"))
df1.write.mode('overwrite').saveAsTable(f"{catalog}.{schema}.product_metadata")

In [0]:
input_path="/Volumes/dev/naval/raw/drivers.json"
catalog="dev"
schema="aryan"
tablename="drivers"

In [0]:
from pyspark.sql.functions import col, current_timestamp

# Extract
df = spark.read.format("json").load(input_path)

# Transform
df1 = df.withColumn("ingestion_date", current_timestamp()) \
        .withColumn("forename", col("name.forename")) \
        .withColumn("surname", col("name.surname")) \
        .withColumn("path", col("_metadata.file_path")) \
        .drop("name")

# Load
df1.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.{tablename}")